# 🔩 Steel Defect Classification (Multiclass) with the NEU Surface Defect Database

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samtzai/course_ai_for_industrial_control_students/blob/main/notebooks/03_steel_defects_classification/n13_steel_defect_neu_multiclass.ipynb)

> **En Colab:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU** (el notebook también funciona en CPU, solo tarda más).

In notebook **n12** we decided *critical defect vs. other defect*. On a real line we also want to know **which** defect it is,
because each type points to a different cause in the process (e.g. `rolled-in_scale` → descaling, `scratches` → rolls or guides).
Here we train a CNN to classify the **six NEU defect types**: `crazing`, `inclusion`, `patches`, `pitted_surface`, `rolled-in_scale`, `scratches`.

📚 Theory: CNNs (Parte 3, slides 21–46), cross-entropy and softmax (Parte 3, slides 19 and 49), multiclass metrics (Parte 2, slide 70).

### What changes from binary to multiclass?
| | Binary (n12) | Multiclass (n13) |
|---|---|---|
| Output layer | 1 neuron (logit) | 6 neurons (one logit per class) |
| Probability | sigmoid | softmax (inside the loss) |
| Loss | `BCEWithLogitsLoss` | `CrossEntropyLoss` |
| Decision | `p > threshold` | `argmax` of the 6 scores |
| Confusion matrix | 2×2 | 6×6 |

In [ ]:
# ⚙️ Setup cell (run first). Detects Google Colab and installs only what may be missing there.
# Locally (uv / VS Code) this cell does nothing: all dependencies are already in pyproject.toml.
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub', 'gdown'], check=False)
print('Running in Colab' if IN_COLAB else 'Running locally')

In [ ]:
# 📦 Import Required Libraries
import os
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')

QUICK_RUN = os.environ.get('QUICK_RUN', '0') == '1'   # only for automatic tests
IMG_SIZE = 128
EPOCHS = 4 if QUICK_RUN else 25
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
print('QUICK_RUN =', QUICK_RUN, '| EPOCHS =', EPOCHS)

In [ ]:
# 📥 Download the NEU Surface Defect Database into ./data
# Sources tried in order (no Kaggle login needed for the first two):
#   1) kagglehub: 'kaustubhdikshit/neu-surface-defect-database' (public Kaggle mirror of NEU-DET)
#   2) Official Google Drive link of NEU-DET published by the dataset authors (via gdown)
#   3) Manual: download NEU-DET (or NEU-CLS) and unzip it anywhere inside ./data
import os
import zipfile
from pathlib import Path
from PIL import Image

DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)
os.environ.setdefault('KAGGLEHUB_CACHE', str((DATA_DIR / 'kagglehub').resolve()))  # keep everything in ./data

NEU_CLASSES = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
NEU_KAGGLE_HANDLE = 'kaustubhdikshit/neu-surface-defect-database'
NEU_GDRIVE_ID = '1qrdZlaDi272eA79b0uCwwqPrm2Q_WI3k'   # NEU-DET.zip, official link from the authors' page


def list_neu_images(root):
    """All NEU images below `root`. Class and index come from the file name: <class>_<index>.jpg"""
    files = {}
    for p in Path(root).rglob('*.jpg'):
        cls, _, idx = p.stem.rpartition('_')
        if cls in NEU_CLASSES and idx.isdigit():
            files.setdefault(p.name, p)  # de-duplicate by file name
    return sorted(files.values())


def download_neu():
    if len(list_neu_images(DATA_DIR)) >= 1800:
        return DATA_DIR
    errors = []
    try:
        import kagglehub
        path = kagglehub.dataset_download(NEU_KAGGLE_HANDLE)
        if len(list_neu_images(path)) >= 1800:
            return Path(path)
    except Exception as e:  # noqa: BLE001
        errors.append(f'kagglehub: {e}')
    try:
        import gdown
        zip_path = DATA_DIR / 'NEU-DET.zip'
        if not zip_path.exists():
            gdown.download(id=NEU_GDRIVE_ID, output=str(zip_path), quiet=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(DATA_DIR)
        if len(list_neu_images(DATA_DIR)) >= 1800:
            return DATA_DIR
    except Exception as e:  # noqa: BLE001
        errors.append(f'Google Drive: {e}')
    raise RuntimeError(
        'Could not download NEU automatically.\n' + '\n'.join(errors) +
        '\nManual option: download NEU-DET from https://www.kaggle.com/datasets/' + NEU_KAGGLE_HANDLE +
        ' or from http://faculty.neu.edu.cn/songkc/en/zdylm/263270/list/index.htm and unzip it inside ./data')


NEU_ROOT = download_neu()
print(f'NEU images found: {len(list_neu_images(NEU_ROOT))} in {NEU_ROOT}')

In [ ]:
# 🧱 Load the images (grayscale, resized) and build a reproducible train / validation / test split
def load_neu(root, img_size):
    paths = list_neu_images(root)
    X = np.stack([np.array(Image.open(p).convert('L').resize((img_size, img_size), Image.BILINEAR))
                  for p in paths])                                       # (N, H, W) uint8
    y = np.array([NEU_CLASSES.index(p.stem.rpartition('_')[0]) for p in paths])
    idx = np.array([int(p.stem.rpartition('_')[2]) for p in paths])     # image number inside its class
    return X, y, idx


def blocked_split(y, idx, block_size=10, fractions=(0.6, 0.2, 0.2), seed=42):
    """Stratified split by blocks of consecutive image numbers.
    NEU images are crops of the same steel strips and neighbouring numbers can be very similar,
    so we keep each block of `block_size` consecutive images in ONE split (avoids optimistic test results)."""
    rng = np.random.default_rng(seed)
    parts = {'train': [], 'val': [], 'test': []}
    for c in np.unique(y):
        members = np.where(y == c)[0]
        blocks = (idx[members] - 1) // block_size
        order = rng.permutation(np.unique(blocks))
        n_tr = int(round(fractions[0] * len(order)))
        n_va = int(round(fractions[1] * len(order)))
        for name, sel in [('train', order[:n_tr]), ('val', order[n_tr:n_tr + n_va]), ('test', order[n_tr + n_va:])]:
            parts[name].extend(members[np.isin(blocks, sel)])
    return tuple(np.array(sorted(parts[k])) for k in ('train', 'val', 'test'))


X_all, y_all, idx_all = load_neu(NEU_ROOT, IMG_SIZE)
train_idx, val_idx, test_idx = blocked_split(y_all, idx_all, seed=SEED)
print('All images:', X_all.shape, '| classes:', NEU_CLASSES)
for name, ids in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    print(f'{name:>5}: {len(ids):4d} images | per class: {np.bincount(y_all[ids], minlength=6).tolist()}')

In [ ]:
# 👀 Visualize the six defect types (the same split as in n12 and n14)
fig, axes = plt.subplots(3, 6, figsize=(10, 5.5))
for c, name in enumerate(NEU_CLASSES):
    for r, i in enumerate(train_idx[y_all[train_idx] == c][:3]):
        axes[r, c].imshow(X_all[i], cmap='gray'); axes[r, c].axis('off')
    axes[0, c].set_title(name, fontsize=9)
plt.suptitle('NEU: intra-class variability is large, and some classes look similar (RS / crazing / PS)')
plt.tight_layout()
plt.show()

In [ ]:
# 🧪 Normalise with train statistics and create the DataLoaders
X_float = X_all.astype(np.float32) / 255.0
MEAN, STD = X_float[train_idx].mean(), X_float[train_idx].std()
X_norm = (X_float - MEAN) / STD
num_classes = len(NEU_CLASSES)


def make_loader(ids, shuffle):
    ds = TensorDataset(torch.from_numpy(X_norm[ids]).unsqueeze(1), torch.from_numpy(y_all[ids]).long())
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)


torch.manual_seed(SEED)
train_loader, val_loader, test_loader = make_loader(train_idx, True), make_loader(val_idx, False), make_loader(test_idx, False)
print(f'Batches: train={len(train_loader)} val={len(val_loader)} test={len(test_loader)} | classes={num_classes}')

## 🧠 Build the CNN

Same architecture as in n12 but with **6 outputs**. `CrossEntropyLoss` applies `log_softmax` internally, so the model returns raw scores (logits).

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, kernel_size=3, padding=1), nn.BatchNorm2d(c_out),
                         nn.ReLU(), nn.MaxPool2d(2))


class SteelDefectCNN(nn.Module):
    """4 x (Conv-BN-ReLU-MaxPool) + Global Average Pooling + Dropout + Linear."""
    def __init__(self, num_outputs=6, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, 64), conv_block(64, 128))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(128, num_outputs))

    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))

In [ ]:
torch.manual_seed(SEED)
model = SteelDefectCNN(num_outputs=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(model.classifier)
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 🏋️ Train the model
We keep the weights of the epoch with the **lowest validation loss**.

In [ ]:
def run_epoch(model, loader, train):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            n += len(yb)
    return total_loss / n, correct / n


history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
best_val, best_state, best_epoch = np.inf, None, 0
start = time.time()
for epoch in range(EPOCHS):
    tr_loss, tr_acc = run_epoch(model, train_loader, train=True)
    va_loss, va_acc = run_epoch(model, val_loader, train=False)
    for k, v in zip(history, [tr_loss, tr_acc, va_loss, va_acc]):
        history[k].append(v)
    if va_loss < best_val:
        best_val, best_state, best_epoch = va_loss, copy.deepcopy(model.state_dict()), epoch + 1
    print(f'Epoch {epoch + 1:2d}/{EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f}')
print(f'Training time: {time.time() - start:.1f} s | best epoch (min val loss): {best_epoch}')
_ = model.load_state_dict(best_state)

In [ ]:
# 📈 Training history
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history['accuracy'], label='train'); axes[0].plot(history['val_accuracy'], label='validation')
axes[1].plot(history['loss'], label='train'); axes[1].plot(history['val_loss'], label='validation')
for ax, title in zip(axes, ['Accuracy', 'Cross-entropy loss']):
    ax.axvline(best_epoch - 1, color='gray', ls='--', label='best epoch')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 📊 Model Evaluation on the test set

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, labels = [], []
    for xb, yb in loader:
        probs.append(F.softmax(model(xb.to(device)), dim=1).cpu().numpy())
        labels.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(labels)


proba_test, y_test = predict(model, test_loader)
y_pred = proba_test.argmax(1)
print(f'Test accuracy: {accuracy_score(y_test, y_pred):.3f}\n')
print(classification_report(y_test, y_pred, target_names=NEU_CLASSES, digits=3))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=NEU_CLASSES, yticklabels=NEU_CLASSES)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion matrix (test)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.show()

In [ ]:
# 🔍 Most frequent confusions and per-class accuracy
off = cm.copy()
np.fill_diagonal(off, 0)
pairs = sorted(((off[i, j], NEU_CLASSES[i], NEU_CLASSES[j]) for i in range(num_classes) for j in range(num_classes) if off[i, j]), reverse=True)
print('Most frequent confusions (count, actual -> predicted):')
for n, a, p in pairs[:5]:
    print(f'  {n:3d}  {a} -> {p}')
per_class = cm.diagonal() / cm.sum(1)
plt.figure(figsize=(7, 3))
plt.bar(NEU_CLASSES, per_class, color='steelblue')
plt.ylim(0, 1.05); plt.ylabel('Recall (per-class accuracy)'); plt.xticks(rotation=30, ha='right')
plt.title('Per-class accuracy on test'); plt.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.show()

In [ ]:
# 🖼️ Sample predictions with confidence (softmax probability of the predicted class)
rng = np.random.default_rng(SEED)
show = rng.choice(len(test_idx), 12, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(12, 4.8))
for ax, j in zip(axes.ravel(), show):
    ax.imshow(X_all[test_idx[j]], cmap='gray'); ax.axis('off')
    ok = y_pred[j] == y_test[j]
    ax.set_title(f'True: {NEU_CLASSES[y_test[j]]}\nPred: {NEU_CLASSES[y_pred[j]]} ({proba_test[j].max():.2f})',
                 fontsize=8, color='green' if ok else 'red')
plt.suptitle('Sample predictions (green = correct, red = wrong)')
plt.tight_layout()
plt.show()

## 📝 Summary and Next Steps

- The same CNN works for 6 classes by changing only the **output layer** and the **loss**.
- The **confusion matrix** tells you *which* defects are mixed up; per-class recall matters more than the global accuracy when some defect types are more critical than others.
- NEU is small (180 training images per class). In **n14** we test whether **data augmentation** helps with the **same split**, and in **n17** whether a **pre-trained network (fine-tuning)** helps when we have even fewer images.

For the quiz you will need: your test accuracy, the two most confused classes in **your** confusion matrix and an explanation of why they may look alike.